# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR² dataset (ordered logistic regression predictors of knowledge adoption in Kenyan rangeland practices) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s in the dataset.

In [ ]:
# List record sets with their @id and field count
if hasattr(metadata, 'recordSet'):
    record_sets = metadata.recordSet
    print(f"Found {len(record_sets)} record set(s).\n")
    for rs in record_sets:
        recset_id = getattr(rs, '@id', None)
        name = getattr(rs, 'name', None)
        fields = getattr(rs, 'field', [])
        print(f"RecordSet @id: {recset_id} | Name: {name} | Number of fields: {len(fields)}")
        for field in fields:
            field_id = getattr(field, '@id', None)
            field_name = getattr(field, 'name', None)
            col_id = getattr(field, 'column', None)
            print(f"  Field @id: {field_id} | Name: {field_name} | Column: {col_id}")
else:
    print("No record sets found in the dataset metadata.")

## 3. Data Extraction
Load data from all available record sets into DataFrames for analysis. All access references entities by their `@id`.

In [ ]:
# Retrieve the @id of all record sets
record_set_ids = []
if hasattr(metadata, 'recordSet'):
    for rs in metadata.recordSet:
        recset_id = getattr(rs, '@id', None)
        if recset_id:
            record_set_ids.append(recset_id)
else:
    print("No record sets are defined in the metadata.")

print(f"Record set @ids available: {record_set_ids}")

# Load records from all record sets
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded data for record set '{record_set_id}'. Number of records: {df.shape[0]}. Fields: {list(df.columns)}")
    except Exception as e:
        print(f"Could not load records for record set {record_set_id}: {e}")

# Display sample records from the first available record set
if record_set_ids:
    first_rs_id = record_set_ids[0]
    if first_rs_id in dataframes:
        print(f"\nColumns for {first_rs_id}: {dataframes[first_rs_id].columns.tolist()}")
        display(dataframes[first_rs_id].head())
    else:
        print(f"No data loaded for record set {first_rs_id}.")

## 4. Exploratory Data Analysis (EDA)
Apply some common data processing steps, such as filtering records based on a numeric field, normalizing data, or grouping. Entities are always referenced by their `@id` fields.

In [ ]:
# For demonstration, select the first record set and choose numeric/group fields by @id where possible
import numpy as np

if record_set_ids and dataframes:
    recset_id = record_set_ids[0]
    df = dataframes[recset_id]
    print(f"Exploring DataFrame for record set {recset_id} (rows: {len(df)}):\n")
    
    # Identify numeric columns by pandas dtype
    numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
    non_numeric_columns = df.select_dtypes(exclude=[np.number]).columns.tolist()
    
    if numeric_columns:
        # Just picking the first numeric field for filtering and normalizing
        numeric_field_id = numeric_columns[0]
        print(f"Using numeric field (@id): {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if np.isfinite(df[numeric_field_id].mean()) else 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where '{numeric_field_id}' > {threshold:.2f} (n={len(filtered_df)}):")
        display(filtered_df.head())
        
        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized '{numeric_field_id}' added as '{norm_col}':")
        display(filtered_df[[numeric_field_id, norm_col]].head())
    else:
        print("No numeric columns available for filtering/normalization.")
    
    # Group by a categorical field (first non-numeric column)
    group_field_id = non_numeric_columns[0] if non_numeric_columns else None
    if group_field_id and numeric_columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by '{group_field_id}', mean of '{numeric_field_id}':")
        display(grouped_df.head())
    else:
        print("No categorical column found for grouping.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships in the dataset.

In [ ]:
# Simple histogram of the selected numeric field
import matplotlib.pyplot as plt

if record_set_ids and dataframes:
    recset_id = record_set_ids[0]
    df = dataframes[recset_id]
    numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_columns:
        numeric_field_id = numeric_columns[0]
        plt.figure(figsize=(8, 5))
        plt.hist(df[numeric_field_id].dropna(), bins=20, color='skyblue', edgecolor='black')
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel('Count')
        plt.show()
    else:
        print("No numeric columns for histogram visualization.")
else:
    print("No data for visualization.")

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to load and inspect a FAIR² Croissant-formatted dataset. We reviewed its metadata, programmatically explored record sets and fields by their `@id`, loaded them into DataFrames, and performed basic analysis. This approach enables robust, reproducible FAIR dataset pipelines for downstream ML and analysis tasks.